# Analysis of Area Normalization Factor (ANF)

In [ ]:
"""
File Name: compare_anf_baseline.ipynb

Author:
    Alessio Morea
    B-Open Solutions srl

Date: August 2025

Description:
------------
This notebook analyzes the Area Normalization Factor (ANF) for each image in a
time-series stack of Single Look Complex (SLC) data.

Usage:
------
To use this script, define the following parameters:

    - data_path:  Path to the root directory containing the ANF computation results. It is expected that this directory contains a separate subdirectory for each acquisition date, generated by a preceding script (compute_anf_saocom.ipynb).
"""

In [ ]:
import s1_ard_core.computing.io
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib import colormaps
from matplotlib.colors import LogNorm

plt.rcParams.update({"font.size": 20})
plt.rcParams["lines.markersize"] = 12
plt.rcParams["font.family"] = "serif"
cmap = plt.get_cmap("tab10")
colors = [cmap(i) for i in range(cmap.N)]
markers = ["s", "^", "v", "s", "P", "*", "X", "d"]
ls = ["--", "-", "dotted", "-."]

In [ ]:
data_path = "s3://bopen-sar/TEST_AM/SAOCOM_ANF"
files = s1_ard_core.computing.io.glob(f"{data_path}/20*")

In [ ]:
for counter, f in enumerate(files):
    anf = s1_ard_core.computing.io.read(f"{f}/static/data/geo_grid", consolidate_metadata=False).anf
    if counter == 0:
        bbox = list(anf.rio.bounds())
    else:
        bbox_new = anf.rio.bounds()
        bbox[0] = max(bbox[0], bbox_new[0])
        bbox[1] = max(bbox[1], bbox_new[1])
        bbox[2] = min(bbox[2], bbox_new[2])
        bbox[3] = min(bbox[3], bbox_new[3])

In [ ]:
for counter, f in enumerate(files):
    k = f.split("/")[-1]
    print(counter, k)
    anf = s1_ard_core.computing.io.read(f"{f}/static/data/geo_grid", consolidate_metadata=False).anf
    lia = s1_ard_core.computing.io.read(f"{f}/static/data/geo_grid", consolidate_metadata=False).lia
    anf = (
        anf.rio.clip_box(bbox[0], bbox[1], bbox[2], bbox[3])
        .assign_coords({"acquisition_date": k})
        .expand_dims(acquisition_date=1)
        .chunk(1024)
    )
    lia = (
        lia.rio.clip_box(bbox[0], bbox[1], bbox[2], bbox[3])
        .assign_coords({"acquisition_date": k})
        .expand_dims(acquisition_date=1)
        .chunk(1024)
    )
    mask = (anf > 0).compute()
    anf = anf.where(mask, np.nan)
    lia = lia.where(mask, np.nan)
    if counter == 0:
        ANF = anf
        LIA = lia
    else:
        ANF = xr.concat([ANF, anf], dim="acquisition_date")
        LIA = xr.concat([LIA, lia], dim="acquisition_date")

In [ ]:
ANF = ANF.chunk((49, 512, 512))
LIA = LIA.chunk((49, 512, 512))

In [ ]:
mask = np.isfinite(ANF).prod(dim="acquisition_date").compute()

In [ ]:
clean_ANF = ANF.where(mask, np.nan).persist()
clean_LIA = LIA.where(mask, np.nan).persist()

In [ ]:
anf_std = (10 * np.log10(clean_ANF)).std(dim="acquisition_date", skipna=True).compute()
anf_mean = (10 * np.log10(clean_ANF)).mean(dim="acquisition_date", skipna=True).compute()
lia_mean = ((clean_LIA)).mean(dim="acquisition_date", skipna=True).compute()

In [ ]:
anf_ratio = (anf_std / anf_mean).compute()

In [ ]:
(anf_std).drop_vars("spatial_ref").plot.imshow(
    vmin=0, vmax=0.01, figsize=(10, 10), cbar_kwargs={"label": "ANF Std [dB]"}
)

In [ ]:
(anf_mean).drop_vars("spatial_ref").plot.imshow(
    vmin=5.5, vmax=8, figsize=(10, 10), cbar_kwargs={"label": "ANF Mean [dB]"}
)

In [ ]:
(anf_ratio).drop_vars("spatial_ref").plot.imshow(
    vmin=0, vmax=0.0015, figsize=(10, 10), cbar_kwargs={"label": "ANF Std over Mean [dB]"}
)

In [ ]:
h, xedges, yedges = np.histogram2d(
    anf_mean.values.ravel(),
    (anf_std).values.ravel(),
    bins=[np.linspace(-6.5, 13, 1000), np.linspace(0, 0.1, 1000)],
)
h = h.T

In [ ]:
n = 100
plt.figure(figsize=(10, 10))
plt.imshow(h, norm=LogNorm())
plt.xticks([i for i in range(xedges.size)[::n]], labels=[f"{i:.1f}" for i in xedges[::n]], rotation=90)
plt.yticks([i for i in range(yedges.size)[::n]], labels=[f"{i:.2f}" for i in yedges[::n]])
plt.xlabel("ANF mean [dB]")
plt.ylabel("ANF std [dB]")
plt.gca().invert_yaxis()
plt.grid(True, ls="--")

In [ ]:
m = clean_ANF.mean(dim="acquisition_date", skipna=True).squeeze()

In [ ]:
ANF_ratio = (clean_ANF[0] / m).compute()

In [ ]:
h_lia, xedges_lia, yedges_lia = np.histogram2d(
    lia_mean.values.ravel(),
    (ANF_ratio).values.ravel(),
    bins=[np.linspace(0, 1.7, 1000), np.linspace(0.9, 1.1, 1000)],
)
h_lia = h_lia.T

In [ ]:
k = 890
n = 100
plt.figure(figsize=(10, 10))
plt.imshow(h_lia, norm=LogNorm())
# plt.imshow(h_lia, vmin=0, vmax=100)
plt.plot(
    [i for i in range(xedges_lia[:k].size)],
    500 - 10 * np.tan(xedges_lia[:k]),
    color="red",
    ls="--",
    lw=2,
    label=r"$\alpha \cdot \tan \theta_i$",
)
plt.xticks(
    [i for i in range(xedges_lia.size)[::n]], labels=[f"{i:.1f}" for i in np.rad2deg(xedges_lia)[::n]], rotation=90
)
plt.yticks([i for i in range(yedges_lia.size)[::n]], labels=[f"{i:.2f}" for i in yedges_lia[::n]])
plt.xlabel("Local Incidence Angle mean [deg]")
plt.ylabel("ANF / ANF mean")
plt.gca().invert_yaxis()
plt.grid(True, ls="--")
plt.legend(loc="upper left", ncol=1, shadow=True)

In [ ]:
baseline = np.array(
    [
        ["2020-04-23T04:51:27.189461", -1328.0, 999.4025120973221, 1665.6778986050476],
        ["2020-06-10T04:51:24.385192", -1280.0, 263.3277734788695, 656.7543806704007],
        ["2020-07-28T04:51:22.454648", -1232.0, -80.77062822173862, -18.49278494898595],
        ["2020-09-14T04:51:21.932301", -1184.0, -815.8967651520384, -1040.0367101843674],
        ["2020-11-01T04:51:36.008680", -1136.0, 806.7345235407928, 1385.4770490167193],
        ["2020-12-19T04:51:28.364530", -1088.0, -934.5900449356498, -1386.750796387045],
        ["2021-02-05T04:51:32.733324", -1040.0, -47.47230677100682, 0.39487406460100516],
        ["2021-02-21T04:51:28.394290", -1024.0, -539.7488535617833, -775.7805334947851],
        ["2021-03-09T04:51:34.620013", -1008.0, 129.5649202734091, 248.7297462100153],
        ["2021-05-12T04:51:40.487669", -944.0, 336.0405165295723, 680.0314032197397],
        ["2021-06-29T04:51:37.161468", -896.0, -585.7003636471533, -761.2489408150615],
        ["2021-08-16T04:51:37.701698", -848.0, -235.02558329279069, -153.49565160907954],
        ["2021-10-03T04:51:38.769442", -800.0, -310.46686743186575, -329.1246484164686],
        ["2021-10-19T04:51:42.092975", -784.0, 184.33774298465445, 405.0208901971173],
        ["2021-11-04T04:51:43.716983", -768.0, 428.93571815240483, 815.2793684233056],
        ["2021-12-22T04:51:38.790997", -720.0, -499.41653308939794, -552.4189003864485],
        ["2022-02-08T04:51:35.199732", -672.0, -705.4607974961848, -958.4118913238269],
        ["2022-03-28T04:51:38.300844", -624.0, -162.21117150535161, -118.9867924556001],
        ["2022-05-15T04:51:39.681676", -576.0, -313.35051387468104, -280.31691425520745],
        ["2022-07-02T04:51:38.313748", -528.0, -418.2464288237938, -555.2779585328592],
        ["2022-07-26T04:47:51.276456", -504.0, -860.134018482494, -1366.1509580644438],
        ["2022-08-11T04:47:59.540418", -488.0, 671.3429160630955, 1171.4081357161547],
        ["2022-09-12T04:47:58.878700", -456.0, 369.80480436092586, 755.1481386502032],
        ["2022-09-28T04:47:58.420350", -440.0, 124.83591925398059, 264.7528493440299],
        ["2023-03-07T04:47:58.697914", -280.0, -354.66774322939466, -485.0735648994184],
        ["2023-07-21T04:51:46.970444", -144.0, 316.3783349159336, 625.593084688537],
        ["2023-08-14T04:48:04.732890", -120.0, 207.14482045994703, 446.2715752898776],
        ["2023-09-07T04:51:46.271108", -96.0, 273.5933853146744, 512.0433236915923],
        ["2023-10-01T04:48:06.279573", -72.0, -126.62333735813405, -156.25269503633945],
        ["2023-10-25T04:51:45.139087", -48.0, -580.1746373628246, -819.5718176210869],
        ["2023-11-18T04:48:06.677668", -24.0, -301.66540525753794, -424.01643717219076],
        ["2023-12-12T04:51:48.236453", 0.0, 0.0, 0.0],
        ["2024-01-05T04:48:04.559925", 23.0, -370.1353415251379, -512.793310931586],
        ["2024-01-29T04:51:43.419428", 47.0, -384.22042237083645, -478.38899562027916],
        ["2024-02-22T04:48:13.475196", 71.0, 1747.927173491727, 2743.1918986179385],
        ["2024-03-17T04:51:44.729674", 95.0, 113.68576770874938, 212.23847471447078],
        ["2024-04-10T04:48:09.701821", 119.0, 277.7981169876765, 530.0223095519781],
        ["2024-05-04T04:51:43.639739", 143.0, -568.9367093839701, -745.6053381287525],
        ["2024-05-28T04:48:06.980426", 167.0, -38.659058986090514, 14.584390766987976],
        ["2024-06-21T04:51:45.994658", 191.0, 225.21850494731214, 315.45335769393],
        ["2024-07-15T04:48:06.520599", 215.0, 205.8187967262141, 461.5038638438549],
        ["2024-08-08T04:51:44.817443", 239.0, 491.6788293638433, 784.8168951044049],
        ["2024-09-01T04:48:10.081304", 263.0, 797.3752669734295, 1476.5265816288763],
        ["2024-09-25T04:51:44.045440", 287.0, -40.016754891364435, -6.228240984557704],
        ["2024-10-19T04:48:14.351295", 311.0, 1096.3254642328488, 1817.1893692797946],
        ["2024-11-12T04:51:46.196742", 335.0, -88.78407910733311, 18.32309919417234],
        ["2024-12-30T04:51:44.131050", 383.0, 68.58871208896247, 176.62285040821493],
        ["2025-01-23T04:48:05.402821", 407.0, -276.3584070368281, -208.96874025704486],
        ["2025-02-16T04:51:44.026263", 431.0, 260.7798553802659, 705.5972424461731],
    ]
)

In [ ]:
rescaled_ANF = np.abs(10 * np.log10(clean_ANF / clean_ANF.mean(dim=["acquisition_date"], skipna=True)))

In [ ]:
anf_std_time = (rescaled_ANF.std(dim=["y", "x"], skipna=True)).compute()
anf_mean_time = (rescaled_ANF.mean(dim=["y", "x"], skipna=True)).compute()

In [ ]:
r = np.arange(0, 3000, 200)
h = np.arange(0, 0.015, 0.002)
plt.plot(
    abs(baseline[:, -1].astype(float) - np.mean(baseline[:, -1].astype(float))),
    anf_mean_time.values,
    marker="o",
    ls="",
    lw=2,
    alpha=0.7,
    mec="black",
    color=colors[1],
)
plt.grid(True, ls="--")
plt.xticks(r, [f"{i:.0f}" for i in r], rotation=90)
plt.yticks(h, [f"{i:.3f}" for i in h])
plt.xlabel("Orthogonal Baseline [m]")
plt.ylabel(r"$\Delta$ANF [dB]")